# Notebook 03 — RFM Segmentation + CLV Estimation
**Tujuan:** Scoring RFM per customer → 8 segmen strategis → estimasi Customer Lifetime Value.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

OUT     = Path('../output')
FIGURES = OUT / 'figures'

BLUE  = '#2563EB'
RED   = '#DC2626'
GRAY  = '#6B7280'
LIGHT = '#93C5FD'
sns.set_theme(style='whitegrid', font_scale=1.1)

# Palet warna per segmen (8 segmen)
SEG_COLORS = {
    'Champions':          '#1D4ED8',
    'Loyal Customers':    '#2563EB',
    'Big Spenders':       '#7C3AED',
    'Potential Loyalist': '#0891B2',
    'New Customers':      '#059669',
    'At Risk':            '#F59E0B',
    'Hibernating':        '#EF4444',
    'Lost':               '#DC2626',
}

df_orders = pd.read_parquet(OUT / 'df_orders.parquet')
print(f'Loaded: {len(df_orders):,} orders, {df_orders["customer_id"].nunique():,} customers')

## 1. Hitung R, F, M per Customer

In [ ]:
snapshot_date = df_orders['order_date'].max() + pd.Timedelta(days=1)
print(f'Snapshot date: {snapshot_date.date()}')

rfm = (
    df_orders
    .groupby('customer_id')
    .agg(
        last_order_date = ('order_date', 'max'),
        first_order_date= ('order_date', 'min'),
        frequency       = ('order_id', 'nunique'),
        monetary        = ('order_revenue', 'sum')
    )
    .reset_index()
)

rfm['recency'] = (snapshot_date - rfm['last_order_date']).dt.days
rfm['tenure_months'] = (
    (snapshot_date - rfm['first_order_date']).dt.days / 30
).clip(lower=1)  # minimum 1 bulan

print(f'\nRFM shape: {rfm.shape}')
print(rfm[['recency','frequency','monetary']].describe().round(2))

## 2. RFM Scoring (Quintile 1–5)

In [ ]:
def quintile_score(series, reverse=False):
    """Bagi series menjadi 5 quintil. reverse=True untuk Recency (kecil = bagus)."""
    labels = [5, 4, 3, 2, 1] if reverse else [1, 2, 3, 4, 5]
    return pd.qcut(series, q=5, labels=labels, duplicates='drop').astype(int)

rfm['R'] = quintile_score(rfm['recency'],   reverse=True)
rfm['F'] = quintile_score(rfm['frequency'], reverse=False)
rfm['M'] = quintile_score(rfm['monetary'],  reverse=False)

rfm['rfm_score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)

print('Distribusi R score:')
print(rfm['R'].value_counts().sort_index())

## 3. Segmentasi 8 Grup

In [ ]:
def assign_segment(row):
    R, F, M = row['R'], row['F'], row['M']
    if R >= 4 and F >= 4 and M >= 4:
        return 'Champions'
    elif F >= 3 and M >= 3:
        return 'Loyal Customers'
    elif M == 5:
        return 'Big Spenders'
    elif R == 5 and F == 1:
        return 'New Customers'
    elif R >= 3 and F <= 2:
        return 'Potential Loyalist'
    elif R <= 2 and F >= 3 and M >= 3:
        return 'At Risk'
    elif R == 1 and F == 1:
        return 'Lost'
    else:
        return 'Hibernating'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

seg_dist = rfm['segment'].value_counts()
print('Distribusi segmen:')
print(seg_dist)

## 4. CLV Estimation

In [ ]:
# CLV = avg_order_value × purchase_frequency_per_month × avg_lifespan
rfm['avg_order_value']     = rfm['monetary'] / rfm['frequency']
rfm['purchase_freq_month'] = rfm['frequency'] / rfm['tenure_months']
# Gunakan rata-rata lifespan per segmen sebagai proxy customer lifespan
seg_lifespan = rfm.groupby('segment')['tenure_months'].mean()
rfm['avg_lifespan'] = rfm['segment'].map(seg_lifespan)
rfm['clv_estimate'] = rfm['avg_order_value'] * rfm['purchase_freq_month'] * rfm['avg_lifespan']

print('CLV per segmen (rata-rata):')
clv_seg = rfm.groupby('segment')['clv_estimate'].mean().sort_values(ascending=False)
print(clv_seg.round(2))
print(f'\nAvg CLV semua customer: R$ {rfm["clv_estimate"].mean():,.2f}')

## 5. Visualisasi RFM

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- A: Jumlah customer per segmen ---
seg_order = seg_dist.sort_values().index
bar_colors = [SEG_COLORS.get(s, GRAY) for s in seg_order]
axes[0, 0].barh(seg_order, seg_dist[seg_order], color=bar_colors)
axes[0, 0].set_xlabel('Jumlah Customer')
axes[0, 0].set_title('Distribusi Customer per Segmen RFM', fontweight='bold')
for i, v in enumerate(seg_dist[seg_order]):
    pct = v / len(rfm) * 100
    axes[0, 0].text(v + 20, i, f'{v:,} ({pct:.1f}%)', va='center', fontsize=9)
axes[0, 0].set_xlim(0, seg_dist.max() * 1.25)

# --- B: Scatter Recency vs Frequency ---
for seg in rfm['segment'].unique():
    mask = rfm['segment'] == seg
    axes[0, 1].scatter(
        rfm[mask]['recency'], rfm[mask]['frequency'],
        alpha=0.4, s=rfm[mask]['monetary'] / rfm['monetary'].max() * 100 + 10,
        color=SEG_COLORS.get(seg, GRAY), label=seg
    )
axes[0, 1].set_xlabel('Recency (hari sejak order terakhir)')
axes[0, 1].set_ylabel('Frequency (jumlah order)')
axes[0, 1].set_title('Recency vs Frequency\n(ukuran lingkaran = Monetary)', fontweight='bold')
axes[0, 1].legend(fontsize=8, loc='upper right', markerscale=1.5)
# Boundary lines: 90d (active/at-risk) and 365d (lost)
for day, color, label in [(90, '#F59E0B', '90d'), (365, '#DC2626', '365d')]:
    axes[0, 1].axvline(day, color=color, linestyle='--',
                       linewidth=1.3, alpha=0.55)
    axes[0, 1].text(
        day + 5, axes[0, 1].get_ylim()[1] * 0.92,
        label, fontsize=8, color=color, alpha=0.85
    )

# --- C: Avg CLV per segmen ---
clv_seg_sorted = clv_seg.sort_values()
clv_colors = [SEG_COLORS.get(s, GRAY) for s in clv_seg_sorted.index]
axes[1, 0].barh(clv_seg_sorted.index, clv_seg_sorted.values, color=clv_colors)
axes[1, 0].set_xlabel('Avg CLV (BRL)')
axes[1, 0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
axes[1, 0].set_title('Estimated CLV per Segmen', fontweight='bold')
for i, v in enumerate(clv_seg_sorted.values):
    axes[1, 0].text(v + 1, i, f'R${v:,.0f}', va='center', fontsize=9)
axes[1, 0].set_xlim(0, clv_seg_sorted.max() * 1.2)
avg_clv_all = rfm['clv_estimate'].mean()
axes[1, 0].axvline(avg_clv_all, color='#111827', linestyle='--',
                   linewidth=1.8, alpha=0.75, zorder=5)
axes[1, 0].text(
    avg_clv_all + clv_seg_sorted.max() * 0.02,
    len(clv_seg_sorted) - 0.5,
    f'Avg R${avg_clv_all:,.0f}',
    fontsize=8, color='#111827', fontweight='bold', va='center'
)

# --- D: Heatmap korelasi R, F, M ---
corr = rfm[['recency', 'frequency', 'monetary', 'clv_estimate']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, ax=axes[1, 1],
            annot_kws={'size': 11},
            cbar_kws={'shrink': 0.8})
axes[1, 1].set_title('Korelasi R, F, M, CLV', fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=30)
axes[1, 1].tick_params(axis='y', rotation=0)

plt.suptitle('RFM Segmentation Analysis — Olist E-Commerce',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / 'C_rfm_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Tabel Ringkasan Segmen

In [ ]:
summary = (
    rfm.groupby('segment')
    .agg(
        count         = ('customer_id', 'count'),
        pct           = ('customer_id', lambda x: len(x) / len(rfm) * 100),
        avg_recency   = ('recency', 'mean'),
        avg_frequency = ('frequency', 'mean'),
        avg_monetary  = ('monetary', 'mean'),
        avg_clv       = ('clv_estimate', 'mean')
    )
    .round(2)
    .sort_values('avg_clv', ascending=False)
)

print('=== Ringkasan Segmen RFM ===')
print(summary.to_string())

## 7. Simpan df_rfm

In [ ]:
rfm.to_parquet(OUT / 'df_rfm.parquet', index=False)
print(f'Tersimpan: df_rfm.parquet — {len(rfm):,} customers')
print(f'Validasi NaN: {rfm[["R","F","M","segment","clv_estimate"]].isnull().sum().sum()} NaN')